# V1-A Dynamic Re-centering Only

Fair V0-B comparison: same starting portfolio, same fixed order size, same 1,000 USDT gap.  
**Only Dynamic Re-centering is added. Risk Guardrails are OFF. No compounding. Live execution is OFF.**


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os, bisect, heapq, json, base64, time
from datetime import datetime, timezone
import numpy as np, pandas as pd, requests

SYMBOL="BTCUSDT"; INITIAL_CAPITAL=3000.0
GRID_GAP=1000.0; BUY_FEE=SELL_FEE=0.001
V0_FLOOR=38000.0; V0_CEILING=127000.0
V1_FLOOR=27000.0; V1_CEILING=57000.0; RECENTER_TRIGGER_GRIDS=5
V1_REFERENCE=(V1_FLOOR+V1_CEILING)/2; V1_GRIDS=int((V1_CEILING-V1_FLOOR)/GRID_GAP)
RISK_GUARDRAILS_ENABLED=False; LIVE_EXECUTION_ENABLED=False
DATA_DIR="/content/drive/MyDrive/03.Trading/00.Live Trading"
TIMEFRAME="1m"; START_DATE="2024-01-01"; END_DATE="2026-01-01"
V0_EXPECTED_CASH=234.71329888756648
V0_EXPECTED_BTC=0.06533319587441082
V0_EXPECTED_ORDER=58.67832472189169
print("V1-A: Dynamic Re-centering ON | Risk Guardrails OFF")


In [ ]:
def make_grid(floor, ceiling):
    b=np.arange(floor,ceiling,GRID_GAP,dtype=float)
    return pd.DataFrame({"buy":b,"sell":b+GRID_GAP})

V0_GRID=make_grid(V0_FLOOR,V0_CEILING)

def v0_init_table(start_price):
    g=V0_GRID.copy(); seed=g["sell"]>start_price; reserve=~seed
    weight=int(reserve.sum())+float(np.sum(start_price/g.loc[seed,"buy"].to_numpy(float)))
    q=INITIAL_CAPITAL/weight
    g["seed"]=seed
    g["net_btc"]=q/g["buy"]*(1-BUY_FEE)
    g["cost"]=np.where(seed,g["net_btc"]/(1-BUY_FEE)*start_price,0.0)
    if not np.isclose(reserve.sum()*q+g["cost"].sum(),INITIAL_CAPITAL,atol=1e-8):
        raise AssertionError("V0 initialization mismatch")
    return g,q

def regime(ref,rid):
    half=V1_GRIDS//2; floor=ref-half*GRID_GAP; ceiling=ref+(V1_GRIDS-half)*GRID_GAP
    if floor<=0: raise ValueError("Dynamic floor <= 0")
    return {"id":rid,"ref":float(ref),"floor":float(floor),"ceiling":float(ceiling),
            "buys":np.arange(floor,ceiling,GRID_GAP,dtype=float)}

def new_position(tid,rid,ref,t,market_buy,grid_buy,sell,q,kind):
    if kind=="INITIAL_SEED":
        net_target=q/grid_buy*(1-BUY_FEE); gross=net_target/(1-BUY_FEE); cost=gross*market_buy
    else:
        cost=q; gross=cost/market_buy
    fee_btc=gross*BUY_FEE; btc=gross-fee_btc
    gross_sell=btc*sell; sell_fee=gross_sell*SELL_FEE
    return {"id":tid,"rid":rid,"ref":ref,"kind":kind,"status":"OPEN","buy_time":t,
            "buy_price":float(market_buy),"grid_buy":float(grid_buy),"sell":float(sell),
            "sell_time":pd.NaT,"cost":float(cost),"btc":float(btc),"buy_fee_btc":float(fee_btc),
            "sell_fee":float(sell_fee),"net_sell":float(gross_sell-sell_fee),"pnl":np.nan}

def initialize(data):
    t=data.iloc[0]["open_time"]; p0=float(data.iloc[0]["open"]); g,q=v0_init_table(p0)
    cash=INITIAL_CAPITAL; btc=0.0; pos={}; active={}; heap=[]; events=[]; fee=0.0; tid=0
    for r in g.loc[g["seed"]].itertuples(index=False):
        tid+=1; p=new_position(tid,-1,V1_REFERENCE,t,p0,float(r.buy),float(r.sell),q,"INITIAL_SEED")
        cash-=p["cost"]; btc+=p["btc"]; fee+=p["buy_fee_btc"]*p0
        pos[tid]=p; active[p["grid_buy"]]=tid; heapq.heappush(heap,(p["sell"],tid))
        events.append({"time":t,"side":"BUY","tid":tid,"grid":p["grid_buy"],"initial":True})
    init={"source":"FROZEN_V0_B_BASELINE","start_price":p0,"initial_cash":float(cash),
          "initial_btc":float(btc),"normal_order_size_usdt":float(q),
          "initial_sell_positions":int(g["seed"].sum()),"initial_buy_levels":int((~g["seed"]).sum()),
          "initial_btc_allocation_pct":float((INITIAL_CAPITAL-cash)/INITIAL_CAPITAL*100)}
    return cash,btc,pos,active,heap,events,tid,fee,init

def run_v1a(data):
    cash,btc,pos,active,heap,events,tid,buy_fees,init=initialize(data)
    q=init["normal_order_size_usdt"]; rid=0; rg=regime(V1_REFERENCE,rid)
    rec=[]; realized=0.0; sell_fees=0.0; cycles=0; prev=None
    eq=np.empty(len(data)); cash_c=np.empty(len(data)); btc_c=np.empty(len(data)); open_c=np.empty(len(data),int)
    tol=1e-12
    for i,c in enumerate(data.itertuples(index=False)):
        t=c.open_time; o,h,l,cl=map(float,(c.open,c.high,c.low,c.close))
        budget=cash; sold=set()
        while heap and heap[0][0]<=h+tol:
            _,x=heapq.heappop(heap); p=pos[x]
            if p["status"]!="OPEN": continue
            cash+=p["net_sell"]; btc-=p["btc"]; pnl=p["net_sell"]-p["cost"]
            p.update(status="CLOSED",sell_time=t,pnl=float(pnl)); active.pop(p["grid_buy"],None)
            sold.add(p["grid_buy"]); realized+=pnl; sell_fees+=p["sell_fee"]; cycles+=1
            events.append({"time":t,"side":"SELL","tid":x,"grid":p["grid_buy"],"initial":False})
        start=o if prev is None else max(prev,o); levels=rg["buys"]; L=levels.tolist()
        if l<start:
            a=bisect.bisect_left(L,l); b=bisect.bisect_left(L,start)
            for k in range(b-1,a-1,-1):
                gp=float(levels[k])
                if gp in active or gp in sold: continue
                if budget+tol<q: break
                tid+=1; p=new_position(tid,rg["id"],rg["ref"],t,gp,gp,gp+GRID_GAP,q,"GRID_BUY")
                budget-=q; cash-=q; btc+=p["btc"]; buy_fees+=p["buy_fee_btc"]*gp
                pos[tid]=p; active[gp]=tid; heapq.heappush(heap,(p["sell"],tid))
                events.append({"time":t,"side":"BUY","tid":tid,"grid":gp,"initial":False})
        eq[i]=cash+btc*cl; cash_c[i]=cash; btc_c[i]=btc; open_c[i]=len(active)
        trigger=RECENTER_TRIGGER_GRIDS*GRID_GAP
        if cl>=rg["ref"]+trigger or cl<=rg["ref"]-trigger:
            nr=float(np.floor(cl/GRID_GAP+0.5)*GRID_GAP)
            if nr!=rg["ref"]:
                old=rg["ref"]; rid+=1; rg=regime(nr,rid)
                rec.append({"decision_time":t,"effective_time":data.iloc[i+1]["open_time"] if i+1<len(data) else pd.NaT,
                            "direction":"UP" if nr>old else "DOWN","close":cl,
                            "old_reference":old,"new_reference":nr,"new_floor":rg["floor"],"new_ceiling":rg["ceiling"]})
        prev=cl
    curve=pd.DataFrame({"open_time":data["open_time"],"close":data["close"],"cash":cash_c,
                        "btc":btc_c,"open_positions":open_c,"equity":eq})
    return {"data":data,"curve":curve,"positions":pos,"events":pd.DataFrame(events),"init":init,
            "recenter":rec,"cycles":cycles,"realized":float(realized),
            "buy_fees":float(buy_fees),"sell_fees":float(sell_fees),
            "order_size":float(q),"final_cash":float(cash),"final_btc":float(btc)}


In [ ]:
def load_data():
    f=os.path.join(DATA_DIR,f"{SYMBOL}-{TIMEFRAME}-combined.csv"); d=pd.read_csv(f)
    d["open_time"]=pd.to_datetime(d["open_time"],utc=True)
    d[["open","high","low","close","volume"]]=d[["open","high","low","close","volume"]].astype(float)
    s=pd.Timestamp(START_DATE,tz="UTC"); e=pd.Timestamp(END_DATE,tz="UTC")
    return d.drop_duplicates("open_time").sort_values("open_time").loc[lambda x:(x.open_time>=s)&(x.open_time<e)].reset_index(drop=True)

def stats(data,equity):
    e=np.asarray(equity,float); peak=np.maximum.accumulate(e); dd=e/peak-1
    final=float(e[-1]); ret=final/INITIAL_CAPITAL-1
    days=(data.open_time.iloc[-1]-data.open_time.iloc[0]).total_seconds()/86400
    ann=float(np.expm1(np.log(final/INITIAL_CAPITAL)*(365.25/days)))
    mdd=float(dd.min()); cal=float(ann/abs(mdd)) if mdd<0 else np.nan
    return {"final_equity":final,"net_return":ret,"annualized_return":ann,"max_drawdown":mdd,"calmar_ratio":cal}

def trade_history(r):
    fc=float(r["data"].iloc[-1]["close"]); ft=r["data"].iloc[-1]["open_time"]; rows=[]
    for tid in sorted(r["positions"]):
        p=r["positions"][tid]; closed=p["status"]=="CLOSED"
        rows.append({"Trade ID":tid,"Regime ID":p["rid"],"Entry Type":p["kind"],"Status":p["status"],
                     "Buy Time":p["buy_time"],"Buy Price":p["buy_price"],"Grid Buy Price":p["grid_buy"],
                     "Reference at Buy":p["ref"],"Sell Target":p["sell"],"Sell Time":p["sell_time"],
                     "Order Size (USDT)":p["cost"],"Net P&L":float(p["pnl"]) if closed else p["btc"]*fc-p["cost"],
                     "Holding Time":(p["sell_time"] if closed else ft)-p["buy_time"]})
    return pd.DataFrame(rows)

def audit(r):
    init=r["init"]; curve=r["curve"]; ps=r["positions"]; ev=r["events"]; fc=float(r["data"].iloc[-1]["close"])
    opened=[p for p in ps.values() if p["status"]=="OPEN"]; closed=[p for p in ps.values() if p["status"]=="CLOSED"]
    nb=ev.loc[ev.side.eq("BUY") & ~ev.initial.fillna(False)]
    checks={
      "v0_baseline_source_confirmed":init["source"]=="FROZEN_V0_B_BASELINE",
      "initial_cash_matches_frozen_v0":bool(np.isclose(init["initial_cash"],V0_EXPECTED_CASH,atol=1e-8)),
      "initial_btc_matches_frozen_v0":bool(np.isclose(init["initial_btc"],V0_EXPECTED_BTC,atol=1e-12)),
      "order_size_matches_frozen_v0":bool(np.isclose(r["order_size"],V0_EXPECTED_ORDER,atol=1e-10)),
      "risk_guardrails_disabled":RISK_GUARDRAILS_ENABLED is False,
      "cash_never_negative":bool((curve.cash>=-1e-8).all()),
      "btc_never_negative":bool((curve.btc>=-1e-12).all()),
      "final_equity_identity":bool(np.isclose(curve.equity.iloc[-1],r["final_cash"]+r["final_btc"]*fc,atol=1e-8)),
      "final_btc_matches_open_positions":bool(np.isclose(r["final_btc"],sum(p["btc"] for p in opened),atol=1e-12)),
      "closed_trade_count_reconciliation":r["cycles"]==len(closed),
      "new_grid_buys_occurred":len(nb)>0}
    return {"status":"PASS" if all(checks.values()) else "FAIL","checks":checks,
            "diagnostics":{"new_grid_buy_count":int(len(nb)),"max_open_positions_observed":int(curve.open_positions.max())}}

data=load_data(); result=run_v1a(data); perf=stats(data,result["curve"].equity.to_numpy(float))
history=trade_history(result); check=audit(result)
fc=float(data.iloc[-1].close); opened=[p for p in result["positions"].values() if p["status"]=="OPEN"]
summary={**perf,"initial_capital":INITIAL_CAPITAL,"completed_cycles":result["cycles"],"open_positions":len(opened),
         "final_cash":result["final_cash"],"final_btc":result["final_btc"],"realized_profit":result["realized"],
         "unrealized_pnl":float(sum(p["btc"]*fc-p["cost"] for p in opened)),
         "total_fee_usdt_equiv":result["buy_fees"]+result["sell_fees"],"recenter_count":len(result["recenter"]),
         "risk_guardrails_enabled":False}
print("=== V1-A Dynamic Re-centering Only ===")
for k,v in summary.items(): print(f"{k}: {v}")
print("\n=== Initialization ==="); [print(f"{k}: {v}") for k,v in result["init"].items()]
print("\n=== Audit ===",check["status"]); [print(f"{k}: {v}") for k,v in check["checks"].items()]
print(check["diagnostics"]); display(history.head(100))


In [ ]:
def js(v):
    if isinstance(v,dict): return {str(k):js(x) for k,x in v.items()}
    if isinstance(v,(list,tuple)): return [js(x) for x in v]
    if isinstance(v,np.ndarray): return [js(x) for x in v.tolist()]
    if isinstance(v,np.integer): return int(v)
    if isinstance(v,np.floating): return float(v) if np.isfinite(v) else None
    if isinstance(v,(pd.Timestamp,datetime)): return None if pd.isna(v) else v.isoformat()
    if isinstance(v,pd.Timedelta): return str(v)
    if v is pd.NaT: return None
    if isinstance(v,float): return v if np.isfinite(v) else None
    return v if isinstance(v,(bool,str,int)) or v is None else str(v)

run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir=f"logs/v1/{run_id}"; summary_path=f"{run_dir}/summary.json"; history_path=f"{run_dir}/trade_history.csv"
payload=js({"log_schema_version":5,"run_info":{"run_id":run_id,"generated_at":datetime.now(timezone.utc).isoformat(),
            "strategy":"V1-A Dynamic Re-centering Only","comparison_design":"Same frozen V0-B start/order; guardrails OFF",
            "live_execution_enabled":False},"v0_initialization":result["init"],
            "v1_config":{"initial_floor":V1_FLOOR,"initial_ceiling":V1_CEILING,"initial_reference":V1_REFERENCE,
            "grid_gap":GRID_GAP,"number_of_grids":V1_GRIDS,"recenter_trigger_grids":RECENTER_TRIGGER_GRIDS,
            "normal_order_size_usdt":result["order_size"],"no_compounding":True,"old_positions_survive_recenter":True},
            "risk_guardrails":{"enabled":False},"backtest":{"symbol":SYMBOL,"timeframe":TIMEFRAME,"start":START_DATE,"end":END_DATE,
            "data_rows":len(data)},"summary":summary,"recenter_events":result["recenter"],"audit":check,
            "trade_history_file":history_path})

def upload(path,text,msg):
    token=userdata.get("GITHUB_TOKEN"); url=f"https://api.github.com/repos/natdanaiii/Trading/contents/{path}"
    headers={"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
    if requests.get(url,headers=headers,params={"ref":"main"},timeout=30).status_code==200: raise FileExistsError(path)
    body={"message":msg,"content":base64.b64encode(text.encode()).decode(),"branch":"main"}
    for a in range(1,4):
        r=requests.put(url,headers=headers,json=body,timeout=30)
        if r.status_code in (200,201): return r.json()["commit"]["sha"]
        if r.status_code==409 and a<3: time.sleep(a); continue
        r.raise_for_status()

print("Run ID:",run_id)
print("Summary upload:",upload(summary_path,json.dumps(payload,indent=2,allow_nan=False),f"Add V1-A summary {run_id}"))
print("History upload:",upload(history_path,history.to_csv(index=False),f"Add V1-A trade history {run_id}"))
